# 项目一：最终评估与项目结论

本项目已完成模型比较。由于数据量较小，随机森林、XGBoost 与逻辑回归的交叉验证 AUC 差异不大；而早期预警需要较强可解释性，并优先减少漏报，因此最终采用逻辑回归和风险阈值 0.3。

本 Notebook 只在封存测试集上评估一次，不再根据结果修改模型或阈值。

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
data_path = Path('projects/01_student_risk_prediction/data/raw/student-mat.csv')
if not data_path.exists():
    data_path = Path.home() / 'solo_work/算法工程师/projects/01_student_risk_prediction/data/raw/student-mat.csv'

df = pd.read_csv(data_path, sep=';')
y = (df['G3'] < 10).astype(int)
X = df.drop(columns=['G1', 'G2', 'G3'])

X_develop, X_test, y_develop, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
numeric_features = X_develop.select_dtypes(include='number').columns.tolist()
categorical_features = X_develop.select_dtypes(exclude='number').columns.tolist()

In [3]:
final_model = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ])),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

# 用全部开发集学习最终预处理与模型参数。
final_model.fit(X_develop, y_develop)

risk_threshold = 0.30
y_test_proba = final_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= risk_threshold).astype(int)

In [4]:
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
final_results = pd.Series({
    'threshold': risk_threshold,
    'roc_auc': roc_auc_score(y_test, y_test_proba),
    'precision_at_risk': precision_score(y_test, y_test_pred),
    'recall_at_risk': recall_score(y_test, y_test_pred),
    'f1_at_risk': f1_score(y_test, y_test_pred),
    'identified_risk_students (TP)': tp,
    'missed_risk_students (FN)': fn,
    'false_alarms (FP)': fp,
}, name='final test result').round(3)
display(final_results.to_frame())

,final test result
threshold,0.300
roc_auc,0.715
precision_at_risk,0.486
recall_at_risk,0.692
f1_at_risk,0.571
identified_risk_students (TP),18.000
missed_risk_students (FN),8.000
false_alarms (FP),19.000


## 项目结论

该模型适合做辅助预警名单，而不是自动决定学生结果。它的限制包括：样本量小、来自特定学校与课程、无法证明特征与成绩之间的因果关系。实际使用前仍需用新学期数据持续验证，并由教师结合具体情况进行干预。